### 1. Imports and Configuration

In [1]:
import torch
import torch.nn            as nn
import torch.nn.functional as F

# -----------------------------------------------------------------------
# Configuration -- mirrors notebook 00. Kept inline for stand-alone
# execution. All architectural constants are defined here.
# -----------------------------------------------------------------------

VOCAB_SIZE      = 15
TOK_PAD         = 14
SEQ_LEN         = 128
EMBED_DIM       = 32
NUM_HEADS       = 2
NUM_LAYERS      = 2
FF_DIM          = 64
DROPOUT         = 0.1
TERNARY_THRESH  = 0.05
SEED            = 42

DEVICE = torch.device('cpu')

torch.manual_seed(SEED)

print(f'Vocab size     : {VOCAB_SIZE}')
print(f'Sequence len   : {SEQ_LEN}')
print(f'Embed dim      : {EMBED_DIM}  |  Heads: {NUM_HEADS}  |  Layers: {NUM_LAYERS}  |  FF: {FF_DIM}')
print(f'Ternary thresh : {TERNARY_THRESH}')
print(f'Device         : {DEVICE}')

Vocab size     : 15
Sequence len   : 128
Embed dim      : 32  |  Heads: 2  |  Layers: 2  |  FF: 64
Ternary thresh : 0.05
Device         : cpu


### 2. FP32 Transformer

In [2]:
# -----------------------------------------------------------------------
# Standard causal transformer with pre-norm (LayerNorm before each
# sub-layer). This is the FP32 baseline against which the ternary
# model is compared. All parameters are 32-bit floats.
# -----------------------------------------------------------------------

class TransformerBlock(nn.Module):
    '''
    Single transformer block with pre-norm attention and feed-forward.

    Accepts a linear_cls argument so the same block definition can be
    used for both the FP32 and ternary models -- the only difference
    between them is whether linear_cls is nn.Linear or TernaryLinear.

    Parameters
    ----------
    embed_dim  : int              Embedding / hidden state dimension.
    num_heads  : int              Number of attention heads.
    ff_dim     : int              Feed-forward inner dimension.
    dropout    : float            Dropout probability.
    linear_cls : type             Linear layer class (nn.Linear or TernaryLinear).
    '''

    def __init__(self, embed_dim, num_heads, ff_dim, dropout, linear_cls=nn.Linear):
        super().__init__()

        # Pre-norm for attention sub-layer
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn  = nn.MultiheadAttention(
            embed_dim,
            num_heads,
            dropout     = dropout,
            batch_first = True,
        )

        # Pre-norm for feed-forward sub-layer
        # linear_cls controls whether these are float or ternary layers
        self.norm2 = nn.LayerNorm(embed_dim)
        self.ff1   = linear_cls(embed_dim, ff_dim)
        self.ff2   = linear_cls(ff_dim,    embed_dim)
        self.act   = nn.GELU()
        self.drop  = nn.Dropout(dropout)

    def forward(self, x, attn_mask=None):
        '''
        Parameters
        ----------
        x         : Tensor   (batch, seq_len, embed_dim)
        attn_mask : Tensor   Causal mask (seq_len, seq_len), upper-triangular -inf.

        Returns
        -------
        Tensor
            (batch, seq_len, embed_dim)
        '''
        # Self-attention with residual
        h    = self.norm1(x)
        h, _ = self.attn(h, h, h, attn_mask=attn_mask, need_weights=False)
        x    = x + self.drop(h)

        # Feed-forward with residual
        h = self.norm2(x)
        h = self.drop(self.act(self.ff1(h)))
        h = self.ff2(h)
        x = x + self.drop(h)

        return x


class TinyTransformer(nn.Module):
    '''
    Causal language model built from TransformerBlock layers.

    Token embeddings and the output projection are always float32.
    The linear_cls argument is forwarded to every TransformerBlock,
    making this class the single definition for both the FP32 and
    ternary variants.

    Parameters
    ----------
    vocab_size  : int    Number of token types.
    seq_len     : int    Maximum sequence length (used for positional embedding).
    embed_dim   : int    Token embedding dimension.
    num_heads   : int    Attention heads per block.
    num_layers  : int    Number of transformer blocks.
    ff_dim      : int    Feed-forward inner dimension.
    dropout     : float  Dropout probability.
    linear_cls  : type   Linear layer class forwarded to TransformerBlock.
    '''

    def __init__(self, vocab_size, seq_len, embed_dim, num_heads,
                 num_layers, ff_dim, dropout, linear_cls=nn.Linear):
        super().__init__()

        # Token + positional embeddings -- always float32
        self.tok_embed = nn.Embedding(vocab_size, embed_dim, padding_idx=TOK_PAD)
        self.pos_embed = nn.Embedding(seq_len,    embed_dim)

        # Transformer blocks with configurable linear layer type
        self.layers = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, ff_dim, dropout, linear_cls)
            for _ in range(num_layers)
        ])

        self.norm   = nn.LayerNorm(embed_dim)

        # Output projection -- always float32 for numerical stability
        self.output = nn.Linear(embed_dim, vocab_size, bias=False)

        self.seq_len = seq_len

        self._init_weights()

    def _init_weights(self):
        '''
        Initialise weights with small normal distribution.
        Biases are zeroed. Embedding tables are standard normal scaled by 0.02.
        '''
        for module in self.modules():
            if isinstance(module, (nn.Linear, nn.Embedding)):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, x):
        '''
        Parameters
        ----------
        x : Tensor   (batch, seq_len) integer token indices.

        Returns
        -------
        Tensor
            (batch, seq_len, vocab_size) raw logits.
        '''
        B, T = x.shape
        pos  = torch.arange(T, device=x.device).unsqueeze(0)   # (1, T)
        h    = self.tok_embed(x) + self.pos_embed(pos)          # (B, T, D)

        # Causal mask: position i may only attend to positions 0..i
        mask = nn.Transformer.generate_square_subsequent_mask(T, device=x.device)

        for layer in self.layers:
            h = layer(h, attn_mask=mask)

        h = self.norm(h)
        return self.output(h)   # (B, T, V)


print('FP32 transformer classes defined.')

FP32 transformer classes defined.


### 3. Ternary Quantization Layer

In [3]:
# -----------------------------------------------------------------------
# TernaryLinear replaces nn.Linear in the feed-forward sub-layers.
# Embedding tables, LayerNorm parameters, and the output projection
# remain float32 throughout -- consistent with the BitNet approach of
# only quantizing the weight-heavy inner layers.
#
# Straight-through estimator (STE):
#   Forward  -- uses ternary weight values {-1, 0, +1}
#   Backward -- gradient passes through as if quantization = identity
#
# The trick: w + (q - w).detach()
#   This equals q in the forward pass (the detach is a no-op for values)
#   and equals w in the backward pass (the detach blocks gradients from
#   flowing through the q branch, so only the w term carries gradient).
# -----------------------------------------------------------------------

class TernaryLinear(nn.Linear):
    '''
    Linear layer with ternary weight quantization and straight-through
    gradient estimation.

    Inherits from nn.Linear so it drops in as a replacement with no
    changes to the calling code. The latent float32 weights are updated
    by the optimizer; the quantized weights are derived on every forward
    pass and never stored explicitly.

    Quantization rule:
      q(w) = +1  if w >  TERNARY_THRESH
      q(w) =  0  if |w| <= TERNARY_THRESH
      q(w) = -1  if w < -TERNARY_THRESH

    Parameters
    ----------
    in_features  : int
    out_features : int
    bias         : bool  (default True)
    '''

    def __init__(self, in_features, out_features, bias=True):
        super().__init__(in_features, out_features, bias)
        # Biases are kept as float32 -- only weight matrices are ternarized

    def quantize(self):
        '''
        Apply the threshold rule to produce ternary weights, then wrap
        with the STE so gradients flow through the latent float weights.

        Returns
        -------
        Tensor
            Weight tensor that equals ternary values forward, float weights
            backward.
        '''
        w = self.weight
        q = torch.zeros_like(w)
        q[w >  TERNARY_THRESH] =  1.0
        q[w < -TERNARY_THRESH] = -1.0

        # STE: w + (q - w).detach()
        # Forward:  evaluates as q  (the detach means (q-w).detach() = q-w numerically)
        # Backward: gradient flows only through w (the detach blocks d/d_w of q branch)
        return w + (q - w).detach()

    def forward(self, x):
        '''
        Parameters
        ----------
        x : Tensor   (..., in_features)

        Returns
        -------
        Tensor
            (..., out_features) computed with quantized weights.
        '''
        return F.linear(x, self.quantize(), self.bias)


# -----------------------------------------------------------------------
# Utility: snapshot the current ternary weight distribution of a model.
# Used during training to track how many weights are -1 / 0 / +1 at
# each checkpoint epoch.
# -----------------------------------------------------------------------

def get_ternary_stats(model):
    '''
    Compute distribution statistics for all TernaryLinear layers in a model.

    Parameters
    ----------
    model : nn.Module   A model containing zero or more TernaryLinear layers.

    Returns
    -------
    dict with keys:
        neg1_frac    -- fraction of weights quantized to -1
        zero_frac    -- fraction of weights quantized to  0
        pos1_frac    -- fraction of weights quantized to +1
        latent_var   -- mean variance of latent float weights across layers
        n_weights    -- total number of quantized weights counted
    '''
    neg1 = zero = pos1 = total = 0
    layer_vars = []

    for m in model.modules():
        if isinstance(m, TernaryLinear):
            w = m.weight.detach()
            q = torch.zeros_like(w)
            q[w >  TERNARY_THRESH] =  1.0
            q[w < -TERNARY_THRESH] = -1.0

            neg1  += (q == -1).sum().item()
            zero  += (q ==  0).sum().item()
            pos1  += (q ==  1).sum().item()
            total += q.numel()
            layer_vars.append(w.var().item())

    if total == 0:
        return {'neg1_frac': 0.0, 'zero_frac': 0.0, 'pos1_frac': 0.0,
                'latent_var': 0.0, 'n_weights': 0}

    return {
        'neg1_frac'  : neg1 / total,
        'zero_frac'  : zero / total,
        'pos1_frac'  : pos1 / total,
        'latent_var' : float(sum(layer_vars) / len(layer_vars)),
        'n_weights'  : total,
    }


# -----------------------------------------------------------------------
# Utility: compute the weight churn between two ternary snapshots.
# Churn is the fraction of weights that changed their quantized value
# (e.g. +1 -> 0 or 0 -> -1) since the previous snapshot.
# High churn = optimizer still fighting the quantization boundary.
# Low churn  = weights have settled into stable ternary values.
# -----------------------------------------------------------------------

def get_ternary_snapshot(model):
    '''
    Capture the current ternary weight values for all TernaryLinear layers.

    Parameters
    ----------
    model : nn.Module

    Returns
    -------
    dict[str, Tensor]
        Maps named-module key to the quantized weight tensor (detached copy).
    '''
    snapshot = {}
    for name, m in model.named_modules():
        if isinstance(m, TernaryLinear):
            w = m.weight.detach()
            q = torch.zeros_like(w)
            q[w >  TERNARY_THRESH] =  1.0
            q[w < -TERNARY_THRESH] = -1.0
            snapshot[name] = q.clone()
    return snapshot


def compute_weight_churn(model, prev_snapshot):
    '''
    Compute the fraction of ternary weights that changed value since
    the previous snapshot.

    Parameters
    ----------
    model         : nn.Module               Current model state.
    prev_snapshot : dict[str, Tensor]       Output of get_ternary_snapshot().

    Returns
    -------
    tuple[float, dict[str, Tensor]]
        (churn_fraction, new_snapshot)
    '''
    curr_snapshot = get_ternary_snapshot(model)
    changed = total = 0

    for name, curr_q in curr_snapshot.items():
        if name in prev_snapshot:
            changed += (curr_q != prev_snapshot[name]).sum().item()
            total   += curr_q.numel()

    churn = changed / total if total > 0 else 0.0
    return churn, curr_snapshot


print('TernaryLinear, get_ternary_stats, get_ternary_snapshot, compute_weight_churn defined.')

TernaryLinear, get_ternary_stats, get_ternary_snapshot, compute_weight_churn defined.


### 4. Model Instantiation and Comparison

In [4]:
# -----------------------------------------------------------------------
# Instantiate both models with identical architecture dimensions.
# The only difference is linear_cls: nn.Linear vs TernaryLinear.
# -----------------------------------------------------------------------

MODEL_KWARGS = dict(
    vocab_size = VOCAB_SIZE,
    seq_len    = SEQ_LEN,
    embed_dim  = EMBED_DIM,
    num_heads  = NUM_HEADS,
    num_layers = NUM_LAYERS,
    ff_dim     = FF_DIM,
    dropout    = DROPOUT,
)

torch.manual_seed(SEED)    # Identical initialisation for a fair comparison
fp32_model    = TinyTransformer(**MODEL_KWARGS, linear_cls=nn.Linear).to(DEVICE)

torch.manual_seed(SEED)    # Same seed so latent weights start identical
ternary_model = TinyTransformer(**MODEL_KWARGS, linear_cls=TernaryLinear).to(DEVICE)


# -----------------------------------------------------------------------
# Parameter accounting -- break down by layer type so it is clear
# which components dominate the parameter budget.
# -----------------------------------------------------------------------

def count_params(model):
    '''
    Count total, trainable, and per-component parameters.

    Parameters
    ----------
    model : nn.Module

    Returns
    -------
    dict
        total, trainable, and per-named-component parameter counts.
    '''
    total      = sum(p.numel() for p in model.parameters())
    trainable  = sum(p.numel() for p in model.parameters() if p.requires_grad)
    components = {}
    for name, module in model.named_children():
        components[name] = sum(p.numel() for p in module.parameters())
    return {'total': total, 'trainable': trainable, 'components': components}


fp32_stats    = count_params(fp32_model)
ternary_stats = count_params(ternary_model)

print('Parameter counts (should be identical -- same architecture):')
print(f'  FP32    total : {fp32_stats["total"]:,}')
print(f'  Ternary total : {ternary_stats["total"]:,}')
print()

print('Component breakdown (FP32 model):')
for comp, count in fp32_stats['components'].items():
    print(f'  {comp:<12}  {count:>6,} params')


# -----------------------------------------------------------------------
# Memory footprint comparison -- FP32 stores all parameters as float32
# (4 bytes each). The ternary model stores latent float32 weights for
# the optimizer PLUS has an effective inference cost of ~1.58 bits per
# quantized weight.
# -----------------------------------------------------------------------

n_ternary_weights = sum(
    m.weight.numel()
    for m in ternary_model.modules()
    if isinstance(m, TernaryLinear)
)

fp32_inference_bytes    = fp32_stats['total'] * 4
ternary_inference_bytes = n_ternary_weights * (1.58 / 8) + (fp32_stats['total'] - n_ternary_weights) * 4
ternary_training_bytes  = fp32_stats['total'] * 4    # Latent weights also float32 during training

print()
print('Memory footprint at inference (approximate):')
print(f'  FP32            : {fp32_inference_bytes / 1024:.1f} KB  (all float32)')
print(f'  Ternary         : {ternary_inference_bytes / 1024:.2f} KB  ({n_ternary_weights:,} weights at 1.58 bits)')
print(f'  Reduction       : {fp32_inference_bytes / ternary_inference_bytes:.1f}x')
print()
print(f'Note: during training both models use ~{ternary_training_bytes / 1024:.1f} KB for weights')
print('(ternary model must keep latent float32 weights for gradient updates)')

Parameter counts (should be identical -- same architecture):
  FP32    total : 22,208
  Ternary total : 22,208

Component breakdown (FP32 model):
  tok_embed        480 params
  pos_embed      4,096 params
  layers        17,088 params
  norm              64 params
  output           480 params

Memory footprint at inference (approximate):
  FP32            : 86.8 KB  (all float32)
  Ternary         : 56.33 KB  (8,192 weights at 1.58 bits)
  Reduction       : 1.5x

Note: during training both models use ~86.8 KB for weights
(ternary model must keep latent float32 weights for gradient updates)


In [5]:
# -----------------------------------------------------------------------
# Sanity forward pass -- verify shapes are correct before training
# and confirm that ternary weights are being applied in the forward pass.
# -----------------------------------------------------------------------

dummy_input = torch.randint(0, VOCAB_SIZE - 1, (4, SEQ_LEN)).to(DEVICE)  # (B=4, T=128)

with torch.no_grad():
    fp32_out    = fp32_model(dummy_input)
    ternary_out = ternary_model(dummy_input)

print(f'Input shape    : {tuple(dummy_input.shape)}')
print(f'FP32 output    : {tuple(fp32_out.shape)}  |  dtype: {fp32_out.dtype}')
print(f'Ternary output : {tuple(ternary_out.shape)}  |  dtype: {ternary_out.dtype}')

# Verify outputs differ -- if identical, TernaryLinear is not actually
# applying quantization
outputs_differ = not torch.allclose(fp32_out, ternary_out)
print(f'Outputs differ  : {outputs_differ}  (expected True -- quantization changes values)')

# Spot-check a TernaryLinear layer -- all quantized weight values should
# be exactly in {-1, 0, +1}
sample_layer = next(m for m in ternary_model.modules() if isinstance(m, TernaryLinear))
with torch.no_grad():
    q_weights   = sample_layer.quantize()
    unique_vals = q_weights.unique().tolist()

print(f'Quantized weight unique values: {unique_vals}  (expected subset of [-1, 0, 1])')

# Initial ternary distribution (most weights near 0 at init, so zero_frac is expected high)
init_stats = get_ternary_stats(ternary_model)
print()
print('Initial ternary weight distribution (right after init):')
print(f'  -1 fraction : {init_stats["neg1_frac"]:.3f}')
print(f'   0 fraction : {init_stats["zero_frac"]:.3f}  (expected high -- small init weights near zero)')
print(f'  +1 fraction : {init_stats["pos1_frac"]:.3f}')

Input shape    : (4, 128)
FP32 output    : (4, 128, 15)  |  dtype: torch.float32
Ternary output : (4, 128, 15)  |  dtype: torch.float32
Outputs differ  : True  (expected True -- quantization changes values)
Quantized weight unique values: [-1.0, 0.0, 1.0]  (expected subset of [-1, 0, 1])

Initial ternary weight distribution (right after init):
  -1 fraction : 0.007
   0 fraction : 0.986  (expected high -- small init weights near zero)
  +1 fraction : 0.007
